# EEG_26 — DANN Cluster-Specific: Domain Adaptation dentro C0 e C1

**Motivazione**: i soggetti si dividono in due fenotipi neurali (C0 fronto-motorio, C1 fronto-occipitale).  
DANN standard su tutti i 74 soggetti deve confondere distribuzioni molto lontane (C0 e C1 hanno topologie diverse).  
Idea: addestrare **due DANN separati**, uno solo su soggetti C0 e uno su soggetti C1,  
riducendo la varianza intra-dominio → il GRL ha un compito più facile, l'encoder si concentra sul segnale parola.

**Architettura DANN**:
```
Input x (B, 61, 384)
  → Encoder (DHSLP-like) → z (B, d_enc)
        ├── TaskClassifier → logits_word (B, 4)        [loss: CE]
        └── GRL → DomainClassifier → logits_subj (B, n_subj)  [loss: CE, gradiente invertito]
```
Loss totale = task_loss - λ * domain_loss  
Il GRL forza l'encoder a produrre feature invarianti al soggetto ma discriminative per la parola.

**Split fenotipi**:
- **C0**: 23 train, 7 val, 7 test  
- **C1**: 27 train, 3 val, 7 test  
⚠️ C1 ha solo 3 soggetti in val — modello C1 potrebbe avere val instabile.

**Confronto atteso**: bAcc DANN_C0, bAcc DANN_C1 vs DHSLP S-Indep (baseline: ~0.24–0.26)  
**Benchmark**: Li et al. 2025 — 78% (loro dataset, diverso)

In [ ]:
import json, logging, re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
import wandb

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg26')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)

# ---- CONFIG ----
N_CHANNELS   = 61
N_SAMPLES    = 384
N_CLASSES    = 4          # concr4
CLUSTER_SCHEME = 'concr4'

# DHSLP encoder params
K_WINDOWS    = 8
N_EDGES      = 32
D_MODEL      = 64
D_ENC        = 128        # dim embedding finale encoder
N_LAYERS     = 2
DROPOUT      = 0.3

# DANN
LAMBDA_MAX   = 1.0        # peso massimo domain loss
LAMBDA_STEPS = 1000       # step per schedule lambda

# Training
LR             = 1e-3
BATCH_SIZE     = 64
MAX_EPOCHS     = 60
PATIENCE       = 12
LABEL_SMOOTHING = 0.1
USE_INSTANCE_NORM = True
DATA_METRIC    = 'abs_pcc'

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k,v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}

T_WIN = N_SAMPLES // K_WINDOWS
log.info(f'project_root: {project_root}')
log.info(f'K_WINDOWS={K_WINDOWS}  T_WIN={T_WIN}  N_EDGES={N_EDGES}  D_MODEL={D_MODEL}')

## §2 — Split fenotipi C0/C1

Cluster labels da `figures/eeg08c_subject_clusters.csv` (colonna `cluster_k2`).

In [ ]:
cluster_df = pd.read_csv(project_root / 'figures' / 'eeg08c_subject_clusters.csv')
C0_ALL = sorted(cluster_df[cluster_df.cluster_k2 == 0].subj_id.tolist())
C1_ALL = sorted(cluster_df[cluster_df.cluster_k2 == 1].subj_id.tolist())

# Split standard (stesso di EEG_13)
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

C0_TRAIN = sorted([s for s in C0_ALL if s in SUBJ_TRAIN])
C0_VAL   = sorted([s for s in C0_ALL if s in SUBJ_VAL])
C0_TEST  = sorted([s for s in C0_ALL if s in SUBJ_TEST])

C1_TRAIN = sorted([s for s in C1_ALL if s in SUBJ_TRAIN])
C1_VAL   = sorted([s for s in C1_ALL if s in SUBJ_VAL])
C1_TEST  = sorted([s for s in C1_ALL if s in SUBJ_TEST])

log.info(f'C0: train={len(C0_TRAIN)} val={len(C0_VAL)} test={len(C0_TEST)}')
log.info(f'C1: train={len(C1_TRAIN)} val={len(C1_VAL)} test={len(C1_TEST)}')
if len(C1_VAL) < 5:
    log.warning('C1 VAL ha solo %d soggetti — val_bacc instabile!', len(C1_VAL))

## §3 — Dataset

Stesso loader di EEG_13: legge da `data/hypergraphs_pruned_abs_pcc/`.  
Aggiunta: `subj_idx` per la domain loss (ID soggetto relativo al cluster, 0-indexed).

In [ ]:
class EEGDANNDataset(Dataset):
    """
    Restituisce (x, word_label, domain_label).
    domain_label = indice soggetto relativo al cluster (0-indexed).
    Serve per la domain loss del DANN.
    """
    def __init__(self, subj_ids, metric=DATA_METRIC, use_instance_norm=True):
        root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
        self.paths, self.word_labels, self.domain_labels = [], [], []
        self.use_instance_norm = use_instance_norm
        # mapping soggetto → indice dominio (0-indexed dentro il cluster)
        subj2domain = {sid: i for i, sid in enumerate(sorted(subj_ids))}
        for p in sorted(root.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m: continue
            sid = int(m.group(1))
            if sid not in subj_ids: continue
            d = torch.load(p, weights_only=False)
            y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            c = label2cluster.get(y_word)
            if c is None: continue
            self.paths.append(p)
            self.word_labels.append(c)
            self.domain_labels.append(subj2domain[sid])
        log.info(f'  {len(self.paths)} trial | {len(subj_ids)} soggetti | {len(set(self.domain_labels))} domini')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        d = torch.load(self.paths[idx], weights_only=False)
        x = d['x'].float()
        if self.use_instance_norm:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        return (x,
                torch.tensor(self.word_labels[idx],   dtype=torch.long),
                torch.tensor(self.domain_labels[idx], dtype=torch.long))


def make_dann_loaders(train_ids, val_ids, test_ids):
    tr = EEGDANNDataset(train_ids)
    va = EEGDANNDataset(val_ids)
    te = EEGDANNDataset(test_ids)
    # weighted sampler su word label per bilanciare classi
    labels   = np.array(tr.word_labels)
    counts   = np.bincount(labels, minlength=N_CLASSES)
    sample_w = torch.tensor(1.0 / counts[labels], dtype=torch.float)
    sampler  = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    kw = dict(num_workers=2, pin_memory=True)
    return (DataLoader(tr, BATCH_SIZE, sampler=sampler, **kw),
            DataLoader(va, BATCH_SIZE, shuffle=False, **kw),
            DataLoader(te, BATCH_SIZE, shuffle=False, **kw),
            len(train_ids))  # n_domains per il domain classifier

## §4 — Architettura DANN

### 4.1 Gradient Reversal Layer (GRL)
Il GRL inverte il gradiente durante il backward pass con peso λ.  
Effetto: l'encoder viene spinto a *confondere* il domain classifier → feature invarianti al soggetto.

### 4.2 Encoder (DHSLP)
Uguale a EEG_13 ma senza la testa di classificazione finale.  
Output: z (B, d_enc) — embedding latente.

### 4.3 Task Classifier
z → 4 classi concr4.

### 4.4 Domain Classifier
GRL(z) → n_subj classi (una per soggetto nel cluster).

In [ ]:
# --- Gradient Reversal Layer ---
class GradientReversalFn(Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lam * grad_output, None


class GRL(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, x, lam=1.0): return GradientReversalFn.apply(x, lam)


def lambda_schedule(step, total_steps=LAMBDA_STEPS, lam_max=LAMBDA_MAX):
    """Lambda cresce da 0 a lam_max seguendo schedule esponenziale (Ganin et al. 2016)."""
    p = step / total_steps
    return lam_max * (2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)


# --- HGNN conv (da EEG_13) ---
class HGNNConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias   = nn.Parameter(torch.zeros(out_ch))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, X, H):
        d_v = H.sum(dim=2).clamp(min=1e-6)
        d_e = H.sum(dim=1).clamp(min=1e-6)
        Dv  = (1.0 / d_v.sqrt()).unsqueeze(-1)
        De  = (1.0 / d_e).unsqueeze(1)
        out = Dv * (X @ self.weight)
        out = torch.bmm(H.transpose(1, 2), out)
        out = De.transpose(1, 2) * out
        out = torch.bmm(H, out)
        return Dv * out + self.bias


# --- Encoder (DHSLP senza testa) ---
class DHSLPEncoder(nn.Module):
    """
    Encoder DHSLP identico a EEG_13 ma senza il classificatore finale.
    Output: z (B, d_enc) — embedding latente da usare nel DANN.
    """
    def __init__(self, n_nodes=N_CHANNELS, T_win=T_WIN, K=K_WINDOWS,
                 n_edges=N_EDGES, d_model=D_MODEL, hidden=D_ENC,
                 n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.K, self.T_win, self.d_model = K, T_win, d_model
        self.E       = nn.Parameter(torch.randn(n_edges, d_model) * 0.01)
        self.pos_enc = nn.Parameter(torch.randn(n_nodes, d_model) * 0.01)
        self.node_proj = nn.Sequential(
            nn.Linear(T_win, d_model), nn.LayerNorm(d_model), nn.ELU()
        )
        dims = [d_model] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(n_layers)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, N, _ = x.shape
        outs = []
        for k in range(self.K):
            x_k  = x[:, :, k*self.T_win:(k+1)*self.T_win]
            feat = self.node_proj(x_k) + self.pos_enc
            scores = torch.matmul(feat, self.E.T) / (self.d_model ** 0.5)
            H_k  = torch.softmax(scores, dim=2)
            out  = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k)
                out = bn(out.reshape(B*N, -1)).reshape(B, N, -1)
                out = F.relu(out)
                out = self.drop(out)
            outs.append(out.mean(dim=1))
        return torch.stack(outs, dim=1).mean(dim=1)  # (B, d_enc)


# --- DANN completo ---
class DANN(nn.Module):
    def __init__(self, n_domains, d_enc=D_ENC):
        super().__init__()
        self.encoder = DHSLPEncoder(hidden=d_enc)
        self.grl     = GRL()
        # Task classifier: parola → 4 classi
        self.task_clf = nn.Sequential(
            nn.Linear(d_enc, d_enc // 2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(d_enc // 2, N_CLASSES)
        )
        # Domain classifier: soggetto → n_domains classi
        self.domain_clf = nn.Sequential(
            nn.Linear(d_enc, d_enc // 2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(d_enc // 2, n_domains)
        )

    def forward(self, x, lam=1.0):
        z           = self.encoder(x)
        task_logits = self.task_clf(z)
        domain_logits = self.domain_clf(self.grl(z, lam))
        return task_logits, domain_logits


# Sanity check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log.info(f'device: {device}')
_m = DANN(n_domains=23).to(device)
_x = torch.randn(4, N_CHANNELS, N_SAMPLES).to(device)
_t, _d = _m(_x, lam=0.5)
assert _t.shape == (4, N_CLASSES) and _d.shape == (4, 23)
n_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
log.info(f'DANN OK — {n_p:,} param')
del _m, _x, _t, _d

## §5 — Training loop DANN

In [ ]:
def run_epoch_dann(model, loader, optimizer=None, global_step=0, total_steps=LAMBDA_STEPS):
    train = optimizer is not None
    model.train() if train else model.eval()
    task_ce  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    domain_ce = nn.CrossEntropyLoss()
    total_task_loss, total_domain_loss = 0.0, 0.0
    all_word_lbl, all_word_pred = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y_word, y_domain in loader:
            x, y_word, y_domain = x.to(device), y_word.to(device), y_domain.to(device)
            lam = lambda_schedule(global_step, total_steps) if train else LAMBDA_MAX
            task_logits, domain_logits = model(x, lam=lam)
            t_loss = task_ce(task_logits, y_word)
            d_loss = domain_ce(domain_logits, y_domain)
            loss   = t_loss + d_loss  # la direzione è già invertita dal GRL
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                global_step += 1
            total_task_loss   += t_loss.item() * len(y_word)
            total_domain_loss += d_loss.item() * len(y_word)
            all_word_lbl.extend(y_word.cpu().numpy())
            all_word_pred.extend(task_logits.argmax(1).cpu().numpy())
    n = len(loader.dataset)
    bacc = balanced_accuracy_score(all_word_lbl, all_word_pred)
    return total_task_loss/n, total_domain_loss/n, bacc, np.array(all_word_lbl), np.array(all_word_pred), global_step


def train_dann(run_name, cluster_name, train_ids, val_ids, test_ids):
    log.info(f'\n=== {run_name} ===')
    tr_l, va_l, te_l, n_domains = make_dann_loaders(train_ids, val_ids, test_ids)
    model = DANN(n_domains=n_domains).to(device)
    n_p   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_steps = MAX_EPOCHS * len(tr_l)

    cfg = dict(
        notebook='EEG_26', model='DANN_DHSLP', cluster=cluster_name,
        n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
        n_domains=n_domains, lambda_max=LAMBDA_MAX,
        k_windows=K_WINDOWS, n_edges=N_EDGES, d_model=D_MODEL, d_enc=D_ENC,
        n_layers=N_LAYERS, dropout=DROPOUT,
        lr=LR, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, patience=PATIENCE,
        use_instance_norm=USE_INSTANCE_NORM, label_smoothing=LABEL_SMOOTHING,
        n_train_subj=len(train_ids), n_val_subj=len(val_ids), n_test_subj=len(test_ids),
        n_params=n_p,
    )
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=run_name, config=cfg, reinit='finish_previous',
                     settings=wandb.Settings(start_method='thread'))

    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    best_val, best_state, patience_cnt, global_step = 0.0, None, 0, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_tl, tr_dl, tr_b, _, _, global_step = run_epoch_dann(model, tr_l, opt, global_step, total_steps)
        va_tl, va_dl, va_b, _, _, _           = run_epoch_dann(model, va_l)
        sched.step()
        lam_now = lambda_schedule(global_step, total_steps)
        run.log({'train/task_loss': tr_tl, 'train/domain_loss': tr_dl, 'train/bacc': tr_b,
                 'val/task_loss':   va_tl, 'val/domain_loss':   va_dl, 'val/bacc':   va_b,
                 'lambda': lam_now, 'epoch': epoch})
        if va_b > best_val:
            best_val = va_b
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE:
            log.info(f'  early stop epoch {epoch}'); break

    model.load_state_dict(best_state)
    _, _, te_b, te_lbl, te_pred, _ = run_epoch_dann(model, te_l)
    run.summary['val_bacc']  = best_val
    run.summary['test_bacc'] = te_b
    run.log({'confusion_matrix': wandb.plot.confusion_matrix(
        preds=te_pred.tolist(), y_true=te_lbl.tolist(),
        class_names=['CONCR', 'AZIONE', 'STATO', 'ASTRATTO'])})
    run.finish()
    log.info(f'  {run_name}: val={best_val:.4f} test={te_b:.4f}')
    return best_val, te_b, te_lbl, te_pred

## §6 — Esperimento C0 (DANN_C0)

DANN addestrato solo su soggetti C0 (23 train, 7 val, 7 test).  
Dominio = soggetto dentro C0 (23 classi).

In [ ]:
log.info('--- DANN C0 ---')
log.info(f'C0 TRAIN: {C0_TRAIN}')
log.info(f'C0 VAL:   {C0_VAL}')
log.info(f'C0 TEST:  {C0_TEST}')

c0_val, c0_test, c0_lbl, c0_pred = train_dann(
    run_name    = f'eeg26_DANN_C0_{CLUSTER_SCHEME}',
    cluster_name= 'C0',
    train_ids   = C0_TRAIN,
    val_ids     = C0_VAL,
    test_ids    = C0_TEST,
)
print(f'\nDANN_C0: val={c0_val:.4f}  test={c0_test:.4f}')

## §7 — Esperimento C1 (DANN_C1)

DANN addestrato solo su soggetti C1 (27 train, 3 val, 7 test).  
⚠️ Solo 3 soggetti in val — val_bacc instabile, non usare per early stopping aggressivo.  
Considera di aumentare PATIENCE a 20 per C1.

In [ ]:
log.info('--- DANN C1 ---')
log.info(f'C1 TRAIN: {C1_TRAIN}')
log.info(f'C1 VAL:   {C1_VAL}  (⚠️ solo {len(C1_VAL)} soggetti)')
log.info(f'C1 TEST:  {C1_TEST}')

# Per C1 con val piccola: aumenta patience
PATIENCE_C1 = 20
_orig_patience = PATIENCE
# patch temporanea — oppure passa come argomento a train_dann

c1_val, c1_test, c1_lbl, c1_pred = train_dann(
    run_name    = f'eeg26_DANN_C1_{CLUSTER_SCHEME}',
    cluster_name= 'C1',
    train_ids   = C1_TRAIN,
    val_ids     = C1_VAL,
    test_ids    = C1_TEST,
)
print(f'\nDANN_C1: val={c1_val:.4f}  test={c1_test:.4f}')

## §8 — Confronto: DANN_C0, DANN_C1 vs DHSLP S-Indep baseline

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

chance = 1 / N_CLASSES

# Baseline EEG_13 (DHSLP S-Indep su tutti i soggetti)
DHSLP_SINDEP_TEST = 0.257   # valore da EEG_13 (aggiornare se rieseguito)

results = {
    'DHSLP\nS-Indep (EEG_13)': DHSLP_SINDEP_TEST,
    f'DANN_C0\n(n_train={len(C0_TRAIN)})': c0_test,
    f'DANN_C1\n(n_train={len(C1_TRAIN)})': c1_test,
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('EEG_26 — DANN Cluster-Specific vs Baseline (S-Indep)', fontsize=13, fontweight='bold')

# Bar chart confronto
ax = axes[0]
colors = ['#90CAF9', '#EF9A9A', '#A5D6A7']
bars = list(results.values())
lbls = list(results.keys())
ax.bar(range(len(bars)), bars, color=colors, alpha=0.9, edgecolor='none')
ax.axhline(chance, color='gray', ls='--', lw=1.5, label=f'Chance ({chance:.0%})')
ax.set_xticks(range(len(lbls)))
ax.set_xticklabels(lbls, fontsize=9)
ax.set_ylabel('Test bAcc')
ax.set_title('Test bAcc — confronto')
ax.legend(fontsize=8)
for i, v in enumerate(bars):
    ax.text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

# Confusion matrix C0
ax2 = axes[1]
if c0_lbl is not None:
    cm = confusion_matrix(c0_lbl, c0_pred, normalize='true')
    ConfusionMatrixDisplay(cm, display_labels=['CONCR','AZIONE','STATO','ASTR']).plot(ax=ax2, colorbar=False)
    ax2.set_title(f'DANN_C0 — test bAcc={c0_test:.3f}')

# Confusion matrix C1
ax3 = axes[2]
if c1_lbl is not None:
    cm = confusion_matrix(c1_lbl, c1_pred, normalize='true')
    ConfusionMatrixDisplay(cm, display_labels=['CONCR','AZIONE','STATO','ASTR']).plot(ax=ax3, colorbar=False)
    ax3.set_title(f'DANN_C1 — test bAcc={c1_test:.3f}')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg26_dann_cluster_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Riepilogo ---')
for name, val in results.items():
    delta = val - DHSLP_SINDEP_TEST
    print(f'  {name.replace(chr(10)," "):30s}: {val:.4f}  (Δ vs baseline: {delta:+.4f})')

## §9 — Ablation λ (opzionale)

Testa valori fissi di λ per vedere se la domain adaptation aiuta o danneggia.  
λ=0 → nessuna adversarial (modello puro task), λ=1 → piena adversarial.

**Esegui solo se §6/§7 danno risultati interessanti.**

In [ ]:
# Ablation λ su C1 (più probabile trovare segnale)
# Decommentare per eseguire

# LAMBDAS = [0.0, 0.1, 0.5, 1.0, 2.0]
# results_lam = {}
# tr_l, va_l, te_l, n_domains = make_dann_loaders(C1_TRAIN, C1_VAL, C1_TEST)
# for lam_fixed in LAMBDAS:
#     # Usa LAMBDA_MAX fisso invece di schedule
#     model = DANN(n_domains=n_domains).to(device)
#     # ... training con lam fisso ...
#     results_lam[lam_fixed] = te_b
# print(results_lam)

print('Ablation λ non eseguita — decommentare il blocco sopra.')

## §10 — Discussione e conclusioni

**Domande di interpretazione**:
1. DANN_C1 > DANN_C0? → coerente con C1 avere proficiency come tratto stabile
2. DANN > DHSLP baseline? → la separazione fenotipica aiuta la generalizzazione cross-soggetto
3. Se DANN ~ baseline → ε²(parola)=0.03 è il vero collo di bottiglia, non la variabilità soggetto

**Per la tesi** (Cap. 4 / Cap. 5):
- DANN cluster-specific come variante del B03 della roadmap, condizionata ai fenotipi
- Se DANN_C1 > baseline: evidenza che la separazione fenotipica aiuta la generalizzazione
- Se DANN ~ baseline: rinforza l'argomento che il limite è nel segnale parola (ε²=0.03), non nella distribuzione soggetto